In [128]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

In [129]:
df = pd.read_csv("../data/titanic3.csv")

In [130]:
df.head(2)

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.00,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.92,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"


Keeping only the required columns

In [131]:
df = df[['pclass', 'sex', 'age', 'sibsp','parch', 'fare', 'embarked', 'survived']]

In [132]:
df.head(2)

,pclass,sex,age,sibsp,parch,fare,embarked,survived
0,1,female,29.00,0,0,211.3375,S,1
1,1,male,0.92,1,2,151.5500,S,1


In [133]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    1309 non-null   int64  
 1   sex       1309 non-null   str    
 2   age       1046 non-null   float64
 3   sibsp     1309 non-null   int64  
 4   parch     1309 non-null   int64  
 5   fare      1308 non-null   float64
 6   embarked  1307 non-null   str    
 7   survived  1309 non-null   int64  
dtypes: float64(2), int64(4), str(2)
memory usage: 81.9 KB


Splitting the data in equal proportions by using the 'stratified_split' method we created

In [134]:
from src.splits import stratified_split

X_raw = df.drop(columns=["survived"])
y = df["survived"]

X_train_raw, X_val_raw, X_test_raw, y_train, y_val, y_test = (
    stratified_split(X_raw.to_numpy(), y.to_numpy(), train_frac=0.6, val_frac=0.2, test_frac=0.2, seed=42))

In [135]:
print(X_train_raw.shape, y_train.shape)
print(X_val_raw.shape, y_val.shape)
print(X_test_raw.shape, y_test.shape)

(785, 7) (785,)
(262, 7) (262,)
(262, 7) (262,)


Reverting into DataFrames and Series accordingly

In [136]:
X_train_raw = pd.DataFrame(X_train_raw, columns=X_raw.columns)
X_val_raw = pd.DataFrame(X_val_raw, columns=X_raw.columns)
X_test_raw = pd.DataFrame(X_test_raw, columns=X_raw.columns)

y_train = pd.Series(y_train)
y_test = pd.Series(y_test)
y_val = pd.Series(y_val)

In [137]:
numeric_columns = ["pclass", "age", "sibsp", "parch", "fare"]

for subset in [X_train_raw, X_val_raw, X_test_raw]:
    subset[numeric_columns] = subset[numeric_columns].apply(pd.to_numeric)

y_train = y_train.astype(int)
y_val = y_val.astype(int)
y_test = y_test.astype(int)

There exist three columns that contain null values: age, fare, embarked

The strategies will be:
1. median for 'age'
2. median for 'fare'
3. mode for 'embarked'

In [138]:
age_median = X_train_raw["age"].median()
fare_median = X_train_raw["fare"].median()
embarked_mode = X_train_raw["embarked"].mode()[0]

print(age_median, fare_median, embarked_mode)

29.0 14.5 S


In [139]:
print("Train missing values:", pd.isna(X_train_raw).sum().sum())
print("Val missing values:  ", pd.isna(X_val_raw).sum().sum())
print("Test missing values: ", pd.isna(X_test_raw).sum().sum())

Train missing values: 165
Val missing values:   46
Test missing values:  55


Filling the null values

In [140]:
for subset in [X_train_raw, X_val_raw, X_test_raw]:
    subset["age"] = subset["age"].fillna(age_median)
    subset["fare"] = subset["fare"].fillna(fare_median)
    subset["embarked"] = subset["embarked"].fillna(embarked_mode)

In [141]:
print("Train missing values:", pd.isna(X_train_raw).sum().sum())
print("Val missing values:  ", pd.isna(X_val_raw).sum().sum())
print("Test missing values: ", pd.isna(X_test_raw).sum().sum())

Train missing values: 0
Val missing values:   0
Test missing values:  0


Next, we one-hot encode two columns: 'sex' and 'embarked'

In [142]:
X_train_encoded = pd.get_dummies(X_train_raw, columns=["sex", "embarked"], dtype=int)
X_val_encoded = pd.get_dummies(X_val_raw, columns=["sex", "embarked"], dtype=int)
X_test_encoded = pd.get_dummies(X_test_raw, columns=["sex", "embarked"], dtype=int)

In [143]:
X_train_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 785 entries, 0 to 784
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   pclass      785 non-null    int64  
 1   age         785 non-null    float64
 2   sibsp       785 non-null    int64  
 3   parch       785 non-null    int64  
 4   fare        785 non-null    float64
 5   sex_female  785 non-null    int64  
 6   sex_male    785 non-null    int64  
 7   embarked_C  785 non-null    int64  
 8   embarked_Q  785 non-null    int64  
 9   embarked_S  785 non-null    int64  
dtypes: float64(2), int64(8)
memory usage: 61.5 KB


Ensure that columns are going to be in the same order

In [144]:
X_val_encoded = X_val_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

X_train = X_train_encoded.to_numpy(dtype=float)
X_val = X_val_encoded.to_numpy(dtype=float)
X_test = X_test_encoded.to_numpy(dtype=float)

y_train = y_train.to_numpy(dtype=int)
y_val = y_val.to_numpy(dtype=int)
y_test = y_test.to_numpy(dtype=int)

In [145]:
print("Training shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("Remaining train missing values:", pd.isna(X_train).sum())
print("Remaining val missing values:", pd.isna(X_val).sum())
print("Remaining test missing values:", pd.isna(X_test).sum())

print("Training columns:", X_train_encoded.columns.tolist())
print("Validation columns match:", X_val_encoded.columns.equals(X_train_encoded.columns))
print("Test columns match:", X_test_encoded.columns.equals(X_train_encoded.columns))

Training shape: (785, 10) (785,)
Validation shape: (262, 10) (262,)
Test shape: (262, 10) (262,)
Remaining train missing values: 0
Remaining val missing values: 0
Remaining test missing values: 0
Training columns: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S']
Validation columns match: True
Test columns match: True
